# R11 — Gate 2 복구: 같은 체크포인트에서 256과 512

`unsloth/Meta-Llama-3.1-8B-Instruct` · arm1·arm3 · 시드 42 · 렌즈 `none,agreeable`.
**한 번 학습하고 그 체크포인트에서 256과 512를 둘 다 생성한다.**

**사전등록**: `R11_GATE2_SAME_CKPT.md`. 판정표는 그 문서 §2.4에 있고
**여기서 눈금을 만들지 않는다.**

★**이 파일을 직접 수정하지 말 것** — `src/make_nb_r11.py`가 생성한다.

## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 (또는 P100) 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
비교 가능성 핀: `unsloth==2026.7.6`, `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth==2026.7.6")
pip("transformers==4.56.2", "trl==0.22.2")

from importlib.metadata import version
_installed_unsloth = version("unsloth")
assert _installed_unsloth == "2026.7.6", (
    f"Unsloth 설치 핀 불일치: 기대=2026.7.6 실제={_installed_unsloth}")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT  = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"   # 휘발성 대용량 (머지 모델 16GB/개)

def _input_tree():
    """붙은 데이터셋을 3단계까지 나열 — 경로가 안 맞을 때 진단용."""
    seen = sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))
    return seen or ["(비어 있음 — 붙은 데이터셋이 하나도 없다)"]

# 코드 데이터셋 위치 자동 탐색. 마운트 경로가 /kaggle/input/<slug>/ 이기도 하고
# /kaggle/input/datasets/<owner>/<slug>/ 이기도 해서 하드코딩하지 않는다.
if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[코드 데이터셋 없음] 'branch-1-premise'가 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> branch-1-premise / gen-data-backup 둘 다 추가\n"
            "  (Secrets와 마찬가지로 데이터셋도 노트북마다 따로 붙여야 한다)"
        )
    INPUT_DIR = os.path.dirname(os.path.dirname(hits[0]))
    print("[code] 데이터셋 발견:", INPUT_DIR)
    shutil.copytree(INPUT_DIR, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

# zip(압축)과 이미 풀린 폴더(Kaggle 자동 압축해제) 둘 다 지원
if not os.path.exists("data/arm1/round1.jsonl"):
    zip_cands = (glob.glob("/kaggle/input/**/gen_data_backup.zip", recursive=True)
                 + glob.glob("/kaggle/working/**/gen_data_backup.zip", recursive=True))
    dir_cands = glob.glob("/kaggle/input/**/arm1/round1.jsonl", recursive=True)
    if zip_cands:
        shutil.unpack_archive(zip_cands[0], "data")
        print("[data] 복구(zip):", zip_cands[0])
    elif dir_cands:
        src_root = os.path.dirname(os.path.dirname(dir_cands[0]))
        shutil.copytree(src_root, "data", dirs_exist_ok=True)
        print("[data] 복구(폴더):", src_root)
    else:
        print("[진단] /kaggle/input 아래 실제 구조:")
        for p in _input_tree():
            print("   ", p)
        raise SystemExit(
            "\n[학습 데이터 없음] 'gen-data-backup'이 이 노트북에 안 붙어 있다.\n"
            "  우측 Input 패널 -> Add Input -> gen-data-backup 추가"
        )

print("cwd:", os.getcwd())
print("src:", sorted(os.listdir("src")))
print("data arms:", sorted(os.path.basename(d) for d in glob.glob("data/arm*")))

# R2 코드가 맞는지 게이트 — 하나라도 실패하면 1차 코드가 올라가 있는 것이다
for path, needle in [("src/eval_refusal.py", "schema"),
                     ("src/train_round.py", "frac_ckpt"),
                     ("src/probe_directions.py", "harm_frame"),
                     ("configs/arms.json", "eval_lenses")]:
    assert os.path.exists(path), f"R2 코드 아님: {path} 없음 → Dataset New Version 재업로드 필요"
    if needle:
        assert needle in open(path, encoding="utf-8").read(), \
            f"R2 코드 아님: {path}에 '{needle}' 없음 → Dataset New Version 재업로드 필요"
print("[gate] R2 코드 확인됨")

## 3. 세션 함수 정의

In [ ]:
import json, shutil, subprocess, os

BASE = "unsloth/Meta-Llama-3.1-8B-Instruct"
FULL = " --full"       # R2는 전 라운드 n=313. 1차의 n 불일치가 라운드 비교를 막았다.
MMLU_LIMIT = "30"

def run(cmd: str):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"실패: {cmd}"

def check(path, min_kb=20):
    """per-prompt 레코드가 실제로 쓰였는지 즉시 확인 — 1차의 핵심 실패를 막는 게이트."""
    kb = os.path.getsize(path) / 1024
    print(f"[check] {os.path.basename(path)} = {kb:.0f} KB")
    assert kb > min_kb, f"per-prompt 레코드가 비었다: {path}"

def probe_baseline(seed, harm_frame="judgement"):
    """동결기저(θ_0, ∅). 학습 전에 한 번. 이후 프로브가 이걸 기준으로 읽는다.
       ⚠️기저와 이후 라운드의 harm_frame이 다르면 ρ·D가 무의미해진다 — 같은 값을 쓸 것."""
    run(f"python src/probe_directions.py --model_path {BASE} --arm arm0 --round 0 --seed {seed} "
        f"--harm_frame {harm_frame}")
    return f"results/probe_arm0_r0_s{seed}.json"

def run_session(arm, seed, lenses="none,agreeable,principled",
                probe=False, frac=False, mmlu=True, eval_rounds=(1, 2, 3),
                harm_frame="judgement", mmlu_limit=None, free_ckpt=True,
                probe_rounds=None, max_new_tokens=256, out_suffix="", save_adapters=""):
    """★2026-07-29 확장 3건:
       eval_rounds  — 학습은 순차라 건너뛸 수 없지만 **평가는 건너뛸 수 있다.**
                      비용의 거의 전부가 eval(1렌즈 n=313 = ~1,800s)이므로 라운드3만
                      재면 세션이 ~3.4h 짧아진다. 시드 반복은 라운드별 곡선이 아니라
                      최종 효과만 필요하므로 eval_rounds=(3,)이 맞다.
       harm_frame   — 프로브 v_harm 추출 프레이밍. 기본 judgement(고친 것).
                      raw는 S1·S2 재현 전용이며 κ_harm≡1이라 무정보다.
       mmlu_limit   — None이면 전역 MMLU_LIMIT. 논문이 보고하는 대조는 **동일 570문항
                      (limit=10)** 이므로 새 arm은 10으로 도는 편이 비교 가능하고 3배 싸다.
       probe_rounds — ★R4 신설. 프로브가 `r in eval_rounds`에 묶여 있어서
                      **eval 없이 프로브만** 뜰 수 없었다. 기하의 라운드 궤적을 재려면
                      eval(렌즈당 25분)을 다시 돌릴 이유가 없다 — 시드42 eval은 이미 있다.
                      None이면 eval_rounds를 따른다(기존 동작 그대로).
       max_new_tokens, out_suffix — ★R5 신설. 512-token 평가는 기존 256-token
                      결과와 파일명이 절대 겹치지 않게 접미사를 함께 쓴다."""
    base_dirs = probe_baseline(seed, harm_frame) if probe else None
    pr_rounds = eval_rounds if probe_rounds is None else probe_rounds
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    prev, prev_dir = BASE, None

    for r in [1, 2, 3]:
        ckpt = f"{CKPT_ROOT}/{arm}_r{r}_s{seed}"
        frac_arg = " --frac_ckpt 0.25,0.5" if (frac and r == 1) else ""
        run(f"python src/train_round.py --arm {arm} --round {r} --seed {seed} "
            f"--base_model {prev} --data data/{arm}/round{r}.jsonl "
            f"--output_dir {ckpt}{frac_arg}"
            + (f" --save_adapter_dir {save_adapters}/{arm}_r{r}_s{seed}" if save_adapters else ""))

        # ★R5: 라운드 r 머지본이 생긴 순간 r-1은 죽은 무게다. 원래는 eval/probe/mmlu가
        #   끝난 뒤에야 지웠는데, 4렌즈 eval은 약 2시간이라 그동안 32GB를 붙들고 있었다.
        #   R5 배치 2가 실제로 여기서 디스크를 터뜨렸다(Docker daemon 오류는 그 증상).
        if prev_dir and os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"[disk] 삭제(학습 직후): {prev_dir}")
            prev_dir = None

        # 세분 라운드: 라운드1 학습 중 저장된 어댑터를 원본 BASE 위에 얹어 평가
        if frac and r == 1:
            for f in ["0p25", "0p5"]:
                ad = f"{CKPT_ROOT}/{arm}_s{seed}/adapter_frac{f}"
                if not os.path.exists(ad):
                    print(f"[warn] 세분 체크포인트 없음: {ad}")
                    continue
                rn = "0.25" if f == "0p25" else "0.5"
                run(f"python src/eval_refusal.py --model_path {BASE} --adapter_path {ad} "
                    f"--arm {arm} --round {rn} --seed {seed} --lenses {lenses}{FULL}{eval_args}")
                check(f"results/eval_{arm}_r{f}_s{seed}{out_suffix}.json")
                if probe:
                    run(f"python src/probe_directions.py --model_path {BASE} --adapter_path {ad} "
                        f"--arm {arm} --round {rn} --seed {seed} --baseline_dirs {base_dirs}")

        if r in eval_rounds:
            run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
            check(f"results/eval_{arm}_r{r}_s{seed}{out_suffix}.json")
        else:
            print(f"[skip] eval arm={arm} r={r} — eval_rounds={eval_rounds} (비용 절감)")

        if probe and r in pr_rounds:
            run(f"python src/probe_directions.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --baseline_dirs {base_dirs} --harm_frame {harm_frame}")

        if mmlu and r == 3:
            run(f"bash src/run_mmlu.sh {ckpt} {arm} {r} {mmlu_limit or MMLU_LIMIT}")

        # 디스크 해제는 학습 직후로 옮겼다(R5). 여기서는 다음 라운드 기준만 갱신한다.
        prev, prev_dir = ckpt, ckpt

    # 루프는 '이전' 라운드만 지운다 — 마지막 라운드 머지본 16GB가 그대로 남는다.
    # 한 세션만 돌면 무해하지만, 세션을 이어 돌리면 세션당 16GB씩 쌓여
    # 시간이 아니라 디스크가 먼저 터진다. MMLU는 위 루프 안에서 이미 끝났다.
    if free_ckpt and prev_dir and os.path.exists(prev_dir):
        shutil.rmtree(prev_dir)
        print(f"[disk] 삭제(최종): {prev_dir}")
        return None
    return prev

def run_baseline(seed, lenses="none,agreeable,principled", mmlu_limit=None,
                 mmlu=True, max_new_tokens=256, out_suffix=""):
    """arm0 = 원본 모델. 학습 없음. 페르소나 방아쇠 자체가 무해함을 보이는 통제군.
       mmlu — ★R4 신설. arm0의 limit=10 MMLU는 이미 있다. 다시 돌리면 20분을 버리고
              results/mmlu_arm0_r0/ 에 타임스탬프 파일이 하나 더 쌓인다.
       max_new_tokens, out_suffix — ★R5 신설. 생성 상한과 충돌 없는 파일 접미사."""
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    run(f"python src/eval_refusal.py --model_path {BASE} --arm arm0 --round 0 "
        f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
    check(f"results/eval_arm0_r0_s{seed}{out_suffix}.json")
    if mmlu:
        run(f"bash src/run_mmlu.sh {BASE} arm0 0 {mmlu_limit or MMLU_LIMIT}")
    else:
        print("[skip] MMLU arm0 — 이미 측정됨(limit=10)")

print("정의 완료")

## 4. ★이번 세션 — `BATCH` 하나만 바꾼다

실측 단가(`R11_GATE2_SAME_CKPT.md` §1.1): 8B 학습 3라운드 ≈2,000s ·
렌즈당 생성 @256 = 1,015s · @512 = 1,740s(페르소나 arm은 2,390s).

| BATCH | 내용 | 순수 GPU 추정 |
|---|---|---:|
| **1** | **arm1 + arm3** (기본) | **5h48m** |
| 2 | arm1만 — 세션이 죽었을 때의 복구용 | 2h55m |
| 3 | arm3만 — 세션이 죽었을 때의 복구용 | 2h55m |

⚠️ **한 arm은 반드시 한 세션 안에서 끝난다.** 256과 512 사이에 세션이 끊기면 체크포인트가
죽고 그 arm은 **두 상한이 다른 가중치**가 된다 — 정확히 고치려는 버그다. 그래서 arm을
완결한 뒤에 다음 arm으로 가고, 죽으면 그 arm만 BATCH 2·3으로 다시 돈다.

### 왜 렌즈가 둘인가

4렌즈면 10h09m으로 9h 한도를 넘는다. 3렌즈도 7h57m이라 여유가 1h뿐이다.
**세션 초과는 부분 실패가 아니라 전손이다.** 판정에 쓰는 두 렌즈만 잰다.

### 왜 arm0·arm2를 안 도는가

arm0는 학습이 없어 두 상한 파일이 **이미 있고 바이트 동일 100%**로 검증됐다.
arm2는 원고의 arm-차등 문장에서 0.96% 바닥값일 뿐이고 무학습 바닥은 arm0가 준다.

In [ ]:
import json, os

# ── 리비전 게이트 ────────────────────────────────────────────────────────────
NB_NAME = "nb_r11"
NB_REV = "R11a"
_ds_revisions = json.load(open("configs/arms.json", encoding="utf-8")).get("nb_revisions", {})
_ds_rev = _ds_revisions.get(NB_NAME)
assert _ds_rev == NB_REV, (
    f"리비전 불일치: 노트북={NB_NAME}/{NB_REV} Dataset={_ds_rev}. "
    f"둘 중 하나가 낡았다 — Dataset을 다시 올리거나 노트북을 다시 Import하라")
print(f"[gate] 리비전 {NB_NAME}/{NB_REV} 일치")

UNSLOTH_PIN = "2026.7.6"

# 판정에 쓰는 두 렌즈만. 늘리면 9h를 넘는다(사전등록 §1.3).
LENSES = "none,agreeable"
SEED = 42

# ★이 세션의 요점: **같은 체크포인트**에서 두 상한을 잰다.
CAPS = {"256": 256, "512": 512}
SUFFIX = {"256": "_r11_same_ckpt_256", "512": "_r11_same_ckpt_512"}

# 머지 전에 라운드별 어댑터를 남긴다. r=16 · 7 target module · 8B ⇒ 약 84MB/라운드.
ADAPTER_ROOT = "/kaggle/working/adapters"

from common import env_versions
RUNTIME_ENV = env_versions()
assert RUNTIME_ENV.get("unsloth") == UNSLOTH_PIN, (
    f"Unsloth 실행 환경 불일치: 기대={UNSLOTH_PIN} stamp={RUNTIME_ENV}")
print(f"[gate] 실행 환경 stamp 일치: {RUNTIME_ENV}")

# BASE는 셀 8의 8B 그대로다 — 이 캠페인은 8B 본 결과를 고치는 것이므로 덮어쓰지 않는다.
assert BASE == "unsloth/Meta-Llama-3.1-8B-Instruct", (
    f"BASE가 8B가 아니다: {BASE} — R11은 8B 본 결과의 감사를 다시 하는 세션이다")
print(f"[gate] BASE = {BASE}")

# ★★ 커밋마다 이 값 하나만 바꾼다. ★★
BATCH = 1

BATCHES = {1: ["arm1", "arm3"], 2: ["arm1"], 3: ["arm3"]}
assert BATCH in BATCHES, f"BATCH는 {sorted(BATCHES)} 중 하나여야 한다"

# 접미사가 기존 코호트와 겹치면 정본을 덮어쓴다. 기계가 막는다.
for _k, _s in SUFFIX.items():
    assert _s.startswith("_r11_same_ckpt_"), f"접미사가 R11 코호트를 가리키지 않는다: {_s}"
    assert "env76" not in _s, f"접미사 {_s}가 기존 코호트와 섞인다"
assert len(set(SUFFIX.values())) == 2, "두 상한의 접미사가 같다 — 하나가 다른 하나를 덮어쓴다"


## 5. ★본 실행 — 한 arm을 완결하고 다음으로 간다

데이터는 **기존 arm1·arm3 라운드 데이터를 그대로 쓴다. 재생성하지 않는다.**

한 arm의 순서:

```
학습 r1→r3  →  eval @256  →  eval @512  →  어댑터 회수  →  체크포인트 삭제  →  snap()
                └────────── 같은 /kaggle/tmp/ckpt/{arm}_r3_s42 ──────────┘
```

`free_ckpt=False`가 마지막 체크포인트를 살려 경로를 돌려준다. 그 경로에 512 eval을 한 번
더 걸고, **그 다음에** 지운다. 두 eval 사이에 학습이 끼지 않으므로 가중치가 같다.

In [ ]:
import os, shutil

arms = BATCHES[BATCH]
print(f"R11 BATCH {BATCH}: {arms} x seed {SEED} @ {sorted(CAPS.values())} 토큰")

def snap(tag):
    """세션마다 누적 zip을 떠서 뒤 arm이 실패해도 앞 결과를 회수한다."""
    zp = shutil.make_archive(f"/kaggle/working/r11_results_{tag}", "zip",
                             "/kaggle/working/experiment/results")
    print(f"[snap] {zp} | {os.path.getsize(zp)/1e6:.2f} MB", flush=True)

for arm in arms:
    print(f"\n===== {arm} x seed {SEED} — 학습 + eval@256 =====", flush=True)
    ckpt = run_session(arm, SEED, lenses=LENSES, eval_rounds=(3,),
                       probe=False, mmlu=False,
                       max_new_tokens=CAPS["256"], out_suffix=SUFFIX["256"],
                       free_ckpt=False, save_adapters=ADAPTER_ROOT)
    assert ckpt and os.path.exists(ckpt), (
        f"free_ckpt=False인데 체크포인트가 없다: {ckpt} — 512를 같은 가중치로 못 잰다")

    print(f"\n===== {arm} — 같은 체크포인트에서 eval@512 =====", flush=True)
    run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round 3 "
        f"--seed {SEED} --lenses {LENSES}{FULL} "
        f"--max_new_tokens {CAPS['512']} --out_suffix {SUFFIX['512']}")
    check(f"results/eval_{arm}_r3_s{SEED}{SUFFIX['512']}.json")

    # 두 eval이 끝난 **뒤에** 지운다. 이 순서가 이 세션의 전부다.
    shutil.rmtree(ckpt)
    print(f"[disk] 삭제(두 상한 완료 후): {ckpt}")
    snap(f"batch{BATCH}_{arm}")


## 6. ★결과 회수

**어댑터를 결과와 따로 회수한다.** 이걸 남기는 것이 이 캠페인의 절반이다 —
다음에 렌즈나 상한을 바꿔 재평가할 때 재학습이 필요 없어지고, 재학습이 없으면
같은 가중치가 보장된다.

In [ ]:
import shutil, os, glob

TAG = f"batch{BATCH}"
zp = shutil.make_archive(f"/kaggle/working/r11_results_{TAG}", "zip",
                         "/kaggle/working/experiment/results")
print("zip:", zp, "| size MB:", round(os.path.getsize(zp)/1e6, 2))
for p in sorted(glob.glob("/kaggle/working/experiment/results/*.json")):
    print(f"  {os.path.getsize(p)/1024:7.0f} KB  {os.path.basename(p)}")

if os.path.exists(ADAPTER_ROOT):
    ap = shutil.make_archive(f"/kaggle/working/r11_adapters_{TAG}", "zip", ADAPTER_ROOT)
    print("\nadapters:", ap, "| size MB:", round(os.path.getsize(ap)/1e6, 2))
    for d in sorted(glob.glob(f"{ADAPTER_ROOT}/*")):
        mb = sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)) / 1e6
        print(f"  {mb:7.1f} MB  {os.path.basename(d)}")
else:
    print("\n[warn] 어댑터가 없다 — save_adapters가 전달되지 않았는지 확인할 것")


## 7. 세션이 성공했는지 — ★성립 조건이 먼저다

**같은 arm의 256 산출물과 512 산출물이 같은 `ckpt_fingerprint`를 가져야 한다.**
다르면 세션이 중간에 끊긴 것이고, 그 arm의 결과를 **버리고 다시 돈다**(사전등록 §2.1).
이 검사가 실험 전체의 성립 조건이다.

In [ ]:
import json, os

N = 313
UNSLOTH_PIN = "2026.7.6"
EXPECTED_LENSES = ["none", "agreeable"]

arms = BATCHES[BATCH]
failed = False

for arm in arms:
    paths = {k: f"results/eval_{arm}_r3_s{SEED}{SUFFIX[k]}.json" for k in SUFFIX}
    loaded = {}
    for k, path in paths.items():
        assert os.path.exists(path), f"결과가 없다: {path}"
        d = json.load(open(path, encoding="utf-8"))
        assert d["schema"] == "r2", f"{path}: schema={d['schema']}"
        assert d["n_prompts"] == N, f"{path}: n_prompts={d['n_prompts']}"
        assert d["max_new_tokens"] == CAPS[k], (
            f"{path}: max_new_tokens={d['max_new_tokens']} — {CAPS[k]}여야 한다")
        assert d["lenses"] == EXPECTED_LENSES, f"{path}: lenses={d['lenses']}"
        assert d["out_suffix"] == SUFFIX[k], f"{path}: out_suffix={d['out_suffix']}"
        assert d["env"].get("unsloth") == UNSLOTH_PIN, f"{path}: env={d['env']}"
        loaded[k] = d

    # ★★ 성립 조건 — 두 상한이 같은 가중치에서 나왔는가 ★★
    fps = {k: (loaded[k].get("ckpt_fingerprint") or {}).get("sha256") for k in loaded}
    if not all(fps.values()):
        print(f"[gate] ✗ {arm}: ckpt_fingerprint가 없다 {fps} — eval_refusal.py가 낡았다")
        failed = True
    elif fps["256"] != fps["512"]:
        print(f"[gate] ✗ {arm}: 두 상한의 가중치가 다르다 — 세션이 끊겼다. "
              f"이 arm을 버리고 다시 돈다. 256={fps['256'][:16]}… 512={fps['512'][:16]}…")
        failed = True
    else:
        print(f"[gate] ✓ {arm}: 같은 체크포인트 {fps['256'][:16]}… — 비교 성립")

    # 판정 입력 — 상한 도달률을 두 상한에서 나란히 (사전등록 §2.2)
    for k in ["256", "512"]:
        for lens in EXPECTED_LENSES:
            recs = loaded[k]["conditions"][lens]["records"]
            cap = sum(1 for r in recs if r.get("hit_cap"))
            eos = sum(1 for r in recs
                      if r.get("finish_reason") == "eos_token" and r.get("too_short") is not None)
            print(f"  [cap] {arm} @{k} {lens:10} hit_cap={cap:4}/{len(recs)} "
                  f"({cap/len(recs):.1%})")

assert not failed, "성립 조건 실패 — 위 ✗ 항목을 해결하기 전에는 결과를 해석하지 않는다"
print(f"\n[gate] R11 BATCH {BATCH} 통과 — {len(arms)}개 arm × 2상한")
